# Cross-Domain Deepfake Detection — Full Colab Runner

**Kavan Patel · AIML B1, SIT Pune · Batch 2024–2028**

This notebook trains and evaluates all three configurations for the ablation table:

| Row | Configuration | Description |
|-----|--------------|-------------|
| 1 | RGB + Frequency Fusion | Dual ResNet-18 (naïve baseline) |
| 2 | Frozen CLIP only | No adversarial term |
| 3 | Frozen CLIP + class-cond. adversarial | **Proposed method** |

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Upload your project folder to Google Drive at `MyDrive/deepfake-grl/`
3. Populate the `data/` folder (Cell 4 has a Kaggle helper)

## Cell 1 — GPU check + Google Drive mount

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('⚠️  No GPU! Go to Runtime → Change runtime type → T4 GPU')

from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/deepfake-grl'
os.chdir(PROJECT_ROOT)
print('Working directory:', os.getcwd())

## Cell 2 — Install dependencies

In [ ]:
!pip install -q -r requirements.txt
print('✅ Dependencies installed')

## Cell 3 — Offline sanity tests (no GPU, no data needed)

Confirms GRL math, model shapes, loss masking, fusion model, and full pipeline mechanics are correct before spending GPU time.

In [ ]:
!python -m pytest tests/ -v
print('\n✅ All tests passed — mechanics are correct')

## Cell 4 — Download & organise datasets

### Environment A — StyleGAN faces (Kaggle)
Requires a Kaggle account (free). Upload your `kaggle.json` API token via the Files panel first.

In [ ]:
import os, shutil
from pathlib import Path

# ── Kaggle setup ──────────────────────────────────────────────────────────
!pip install -q kaggle
!mkdir -p ~/.kaggle
# Upload kaggle.json via Files panel on the left, then run:
!cp /content/drive/MyDrive/kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

# ── Download 140k Real and Fake Faces ────────────────────────────────────
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces -p /tmp/140k --unzip

# ── Subsample into data/environment_a_stylegan/ ──────────────────────────
N = 2500   # images per class — adjust if you want more/less

src_real = Path('/tmp/140k/real-vs-fake/real-vs-fake/train/real')
src_fake = Path('/tmp/140k/real-vs-fake/real-vs-fake/train/fake')
dst_real = Path('data/environment_a_stylegan/real')
dst_fake = Path('data/environment_a_stylegan/fake')
dst_real.mkdir(parents=True, exist_ok=True)
dst_fake.mkdir(parents=True, exist_ok=True)

for src, dst in [(src_real, dst_real), (src_fake, dst_fake)]:
    imgs = sorted(src.glob('*.jpg'))[:N]
    for p in imgs:
        shutil.copy(p, dst / p.name)

print(f'Env A — real: {len(list(dst_real.glob("*.jpg")))}  fake: {len(list(dst_fake.glob("*.jpg")))}')
print('✅ Environment A ready')

In [ ]:
# ── Environment B (DiFF) & C (FaceForensics++) ────────────────────────────
# These require manual download from their respective repos (see README.md).
# Once downloaded, upload them to Google Drive and point to the paths below.

# Example (adapt paths as needed):
# !cp -r /content/drive/MyDrive/diff_real   data/environment_b_diff/real
# !cp -r /content/drive/MyDrive/diff_fake   data/environment_b_diff/fake
# !cp -r /content/drive/MyDrive/ff_real     data/environment_c_faceforensics/real
# !cp -r /content/drive/MyDrive/ff_fake     data/environment_c_faceforensics/fake

print('Manually copy Env B and C data, then run the verification cell below.')

In [ ]:
# ── Verify dataset structure ──────────────────────────────────────────────
from pathlib import Path

EXTS = {'.jpg', '.jpeg', '.png', '.webp'}
envs = {
    'Env A (StyleGAN)':      'data/environment_a_stylegan',
    'Env B (DiFF)':          'data/environment_b_diff',
    'Env C (FaceForensics)': 'data/environment_c_faceforensics',
}

all_ok = True
for name, base in envs.items():
    for cls in ['real', 'fake']:
        folder = Path(base) / cls
        count  = len([p for p in folder.glob('*') if p.suffix.lower() in EXTS]) if folder.exists() else 0
        status = '✅' if count >= 100 else '❌'
        if count < 100:
            all_ok = False
        print(f'  {status}  {name}/{cls}: {count} images')

if all_ok:
    print('\n✅ All environments populated — ready to train!')
else:
    print('\n❌ Some folders are missing or have too few images. Fix before training.')

## Cell 5 — Row 1: Train RGB + Frequency Fusion Baseline

Both ResNet-18 branches are fine-tuned (not frozen), so this takes longer than the CLIP runs.

In [ ]:
!python src/train_fusion.py \
    --data-root data \
    --epochs 10 \
    --batch-size 32 \
    --lr 1e-4 \
    --out-dir checkpoints

print('\n✅ Row 1 (Fusion baseline) training complete')

## Cell 6 — Row 2: Train Frozen CLIP (no adversarial)

Only the two small MLP heads train. Fast on a T4 (~8–12 min).

In [ ]:
!python src/train.py \
    --data-root data \
    --epochs 8 \
    --adversarial 0 \
    --out-dir checkpoints

print('\n✅ Row 2 (Frozen CLIP, no adversarial) training complete')

## Cell 7 — Row 3: Train Frozen CLIP + Class-Conditional Adversarial (Proposed)

GRL is ramped from 0 → 1 over training using the DANN schedule.

In [ ]:
!python src/train.py \
    --data-root data \
    --epochs 8 \
    --adversarial 1 \
    --out-dir checkpoints

print('\n✅ Row 3 (Frozen CLIP + adversarial) training complete')

## Cell 8 — Evaluate all three + print ablation table

In [ ]:
# ── Rows 2 & 3 (CLIP models) ─────────────────────────────────────────────
!python src/evaluate.py \
    --data-root data \
    --checkpoint checkpoints/model_baseline.pt \
    --compare    checkpoints/model_adv.pt \
    --jpeg-qualities 30 50 70

print()

# ── Row 1 (Fusion baseline) ───────────────────────────────────────────────
!python src/evaluate_fusion.py \
    --data-root data \
    --checkpoint checkpoints/model_fusion.pt \
    --jpeg-qualities 30 50 70

In [ ]:
# ── Print the full 3-row ablation table ───────────────────────────────────
import json

fusion   = json.load(open('checkpoints/model_fusion.eval.json'))
baseline = json.load(open('checkpoints/model_baseline.eval.json'))
adv      = json.load(open('checkpoints/model_adv.eval.json'))

rows = [
    ('RGB + Freq. Fusion (Row 1)',        fusion),
    ('Frozen CLIP only  (Row 2)',          baseline),
    ('Frozen CLIP + Adversarial (Row 3)', adv),
]

print(f"{'Configuration':<42} {'ID Acc':>8} {'OOD Acc':>8} {'Gap':>8}")
print('-' * 70)
for label, r in rows:
    print(f"{label:<42} {r['id']['accuracy']:>8.4f} {r['ood']['accuracy']:>8.4f} {r['generalization_gap']:>+8.4f}")

print('\n(Gap = ID acc − OOD acc; smaller is better)')

## Cell 9 — Plot ablation bar chart + save PNG

In [ ]:
import json
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.1)

fusion   = json.load(open('checkpoints/model_fusion.eval.json'))
baseline = json.load(open('checkpoints/model_baseline.eval.json'))
adv      = json.load(open('checkpoints/model_adv.eval.json'))

configs  = ['RGB + Freq.\nFusion', 'Frozen CLIP\n(no adv.)', 'Frozen CLIP\n+ Adversarial']
id_acc   = [fusion['id']['accuracy'],  baseline['id']['accuracy'],  adv['id']['accuracy']]
ood_acc  = [fusion['ood']['accuracy'], baseline['ood']['accuracy'], adv['ood']['accuracy']]

x = range(len(configs))
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar([i - 0.18 for i in x], id_acc,  width=0.35, label='ID accuracy',  color='steelblue', alpha=0.9)
ax.bar([i + 0.18 for i in x], ood_acc, width=0.35, label='OOD accuracy', color='tomato',    alpha=0.9)
ax.set_xticks(list(x))
ax.set_xticklabels(configs)
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1.05)
ax.legend()
ax.set_title('Generalization Gap: 3-configuration ablation\n(smaller gap = better cross-domain generalisation)')
plt.tight_layout()
plt.savefig('checkpoints/ablation_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved checkpoints/ablation_chart.png')

## Cell 10 — Launch Gradio demo (public link)

In [ ]:
!python src/app.py --checkpoint checkpoints/model_adv.pt --share